# Buscador Semántico de Películas

**Trabajo Integrador — Procesamiento de Lenguaje Natural · Grupo 2**

Sistema de búsqueda semántica que recibe descripciones libres en lenguaje natural (por ejemplo: *"una película donde el protagonista pierde la memoria y hay viajes en el tiempo"*) y recupera las películas cuya información textual es semánticamente más similar a la consulta.

Se comparan dos enfoques de recuperación de información:

- **Baseline léxico:** TF-IDF + similitud coseno.
- **Enfoque semántico:** embeddings (Sentence-BERT) + similitud coseno.

Ambos enfoques se evalúan cuantitativamente con **Hit Rate@K** y **MRR** sobre un conjunto de consultas con película esperada (ground truth).

## 1. Configuración del entorno

In [76]:
# Dependencias (ejecutar en Google Colab si no están instaladas)
!pip install -q sentence-transformers scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [77]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## 2. Carga del dataset

El dataset contiene 1000 películas obtenidas de *The Movie Database* (TMDB), con título, sinopsis (`overview`), géneros, keywords, elenco (`cast`), director y metadatos numéricos.

In [78]:
import os

# Ruta relativa cuando se ejecuta desde la carpeta Notebook/ del repositorio.
# En Google Colab: subir el CSV al entorno y dejar solo el nombre del archivo.
CSV_PATH = "../data/movies_expanded_1000.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "movies_expanded_1000.csv"

df = pd.read_csv(CSV_PATH)
df.head()

,id,title,original_title,overview,tagline,release_date,runtime,original_language,genres,keywords,cast,directors,popularity,vote_average,vote_count,overview_length,release_year,search_text,search_text_length
0,10576,Psycho II,Psycho II,Norman Bates is declared sane and released fro...,"It's 22 years later, and Norman Bates is comin...",1983-06-03,113,en,"['Horror', 'Mystery', 'Thriller', 'Crime']","['psychopath', 'insanity', 'motel', 'sequel', ...","['Anthony Perkins', 'Meg Tilly', 'Vera Miles',...",['Richard Franklin'],2.7056,6.510,724,204,1983,Psycho II | Norman Bates is declared sane and ...,572
1,10830,Matilda,Matilda,Matilda Wormwood is an brilliant and intellige...,Somewhere inside all of us is the power to cha...,1996-08-02,98,en,"['Comedy', 'Family', 'Fantasy']","['based on novel or book', 'parent child relat...","['Mara Wilson', 'Danny DeVito', 'Rhea Perlman'...",['Danny DeVito'],11.4750,7.174,4785,413,1996,Matilda | Matilda Wormwood is an brilliant and...,1033
2,89,Indiana Jones and the Last Crusade,Indiana Jones and the Last Crusade,"In 1938, an art collector appeals to eminent a...",Have the adventure of your life keeping up wit...,1989-05-24,127,en,"['Adventure', 'Action']","['saving the world', 'nazi', 'holy grail', 've...","['Harrison Ford', 'Sean Connery', 'Denholm Ell...",['Steven Spielberg'],7.2998,7.852,11019,625,1989,"Indiana Jones and the Last Crusade | In 1938, ...",1328
3,12155,Alice in Wonderland,Alice in Wonderland,"Alice, now 19 years old, returns to the whimsi...",You're invited to a very important date.,2010-03-03,108,en,"['Family', 'Fantasy', 'Adventure']","['based on novel or book', 'queen', ""based on ...","['Mia Wasikowska', 'Johnny Depp', 'Anne Hathaw...",['Tim Burton'],10.4469,6.639,14744,139,2010,"Alice in Wonderland | Alice, now 19 years old,...",638
4,11186,Child's Play 2,Child's Play 2,Chucky is reconstructed by a toy factory to di...,Sorry Jack... Chucky's back!,1990-11-09,84,en,['Horror'],"['factory', 'voodoo', 'foster parents', 'faith...","['Alex Vincent', 'Brad Dourif', 'Christine Eli...",['John Lafia'],4.8559,6.359,2005,172,1990,Child's Play 2 | Chucky is reconstructed by a ...,638


## 3. Validación y limpieza

Verificamos dimensiones, valores nulos y duplicados antes de construir la representación textual.

In [79]:
print("Shape:", df.shape)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDuplicados por id:", df.duplicated(subset="id").sum())
print("Duplicados por title:", df.duplicated(subset="title").sum())

Shape: (1000, 19)

Valores nulos por columna:
id                     0
title                  0
original_title         0
overview               0
tagline               86
release_date           0
runtime                0
original_language      0
genres                 0
keywords               0
cast                   0
directors              0
popularity             0
vote_average           0
vote_count             0
overview_length        0
release_year           0
search_text            0
search_text_length     0
dtype: int64

Duplicados por id: 0
Duplicados por title: 4


In [80]:
# Eliminar duplicados por título, conservando la primera aparición
print(f"Filas antes de eliminar duplicados por título: {len(df)}")

# Mostrar los títulos duplicados antes de eliminar
duplicated_titles = df[df.duplicated(subset="title", keep=False)].sort_values("title")
print(f"\nTítulos duplicados encontrados ({duplicated_titles['title'].nunique()}):")
print(duplicated_titles[["title", "id", "release_year", "vote_count"]])

df = df.drop_duplicates(subset="title", keep="first").reset_index(drop=True)

print(f"\nFilas después de eliminar duplicados: {len(df)}")
print(f"Duplicados por título restantes: {df.duplicated(subset='title').sum()}")

Filas antes de eliminar duplicados por título: 1000

Títulos duplicados encontrados (4):
                    title      id  release_year  vote_count
292               Aladdin  420817          2019       10586
584               Aladdin     812          1992       11976
507  Beauty and the Beast  321612          2017       15977
812  Beauty and the Beast   10020          1991       10634
521         Lilo & Stitch  552524          2025        2129
886         Lilo & Stitch   11544          2002        7056
83        The Running Man     865          1987        2819
872       The Running Man  798645          2025        1455

Filas después de eliminar duplicados: 996
Duplicados por título restantes: 0


In [81]:
# Normalización de columnas textuales: nulos -> "", conversión a string y sin espacios laterales
text_cols = ["title", "overview", "genres", "keywords", "cast"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()

print("Columnas textuales normalizadas.")

Columnas textuales normalizadas.


## 4. Representación textual enriquecida

Para cada película construimos un único texto consolidado (`search_text`) que integra título, géneros, keywords, elenco y sinopsis en un formato estructurado y etiquetado. Esta representación es la entrada **tanto del baseline TF-IDF como del modelo de embeddings**, lo que garantiza una comparación justa entre ambos enfoques.

In [82]:
import ast

def build_search_text(row):
    try:
        kw = str(ast.literal_eval(row['keywords'])[:8])
    except (ValueError, SyntaxError):
        kw = row['keywords']
    return (
        f"Title: {row['title']}. "
        f"Genres: {row['genres']}. "
        f"Keywords: {kw}. "
        f"Cast: {row['cast']}. "
        f"Overview: {row['overview']}."
    )

df["search_text"] = df.apply(build_search_text, axis=1)
print(df.loc[0, "search_text"])

Title: Psycho II. Genres: ['Horror', 'Mystery', 'Thriller', 'Crime']. Keywords: ['psychopath', 'insanity', 'motel', 'sequel', 'revenge', 'murder', 'mental illness', 'framed for murder']. Cast: ['Anthony Perkins', 'Meg Tilly', 'Vera Miles', 'Robert Loggia', 'Dennis Franz']. Overview: Norman Bates is declared sane and released from the facility in which he was being held, despite the complaints of Lila Loomis, sister of his most famous victim. Is he really cured, or will he kill again?.


In [83]:
# Verificación de integridad de la representación
assert df["search_text"].isnull().sum() == 0
assert (df["search_text"].str.strip() == "").sum() == 0

movie_texts = df["search_text"].tolist()
print("Películas a vectorizar:", len(movie_texts))

Películas a vectorizar: 996


## 5. Baseline léxico: TF-IDF + similitud coseno

El baseline representa cada película mediante un vector TF-IDF (frecuencia de término ponderada por frecuencia inversa de documento) y recupera resultados por similitud coseno frente al vector de la consulta. Captura coincidencias **léxicas**, no semánticas.

In [84]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(movie_texts)
print("Shape matriz TF-IDF:", tfidf_matrix.shape)

Shape matriz TF-IDF: (996, 5000)


In [85]:
def search_tfidf(query, top_k=5):
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_idx][["title", "overview", "genres", "keywords", "cast"]].copy()
    results["similarity"] = similarities[top_idx]
    return results.reset_index(drop=True)

In [86]:
search_tfidf("movie about dreams, mind and memory", top_k=5)

,title,overview,genres,keywords,cast,similarity
0,The Maze Runner,A teenager with no memory of his past finds hi...,"['Action', 'Mystery', 'Science Fiction', 'Thri...","['based on novel or book', 'escape', 'dystopia...","[""Dylan O'Brien"", 'Kaya Scodelario', 'Thomas B...",0.158704
1,In Your Dreams,Stevie and her little brother Elliot journey i...,"['Adventure', 'Animation', 'Comedy', 'Family',...","['magic', 'wish', 'family', 'dream']","['Jolie Hoang-Rappaport', 'Elias Janssen', 'Si...",0.139933
2,Lola's Secret,Young man has his dreams come true when the se...,"['Drama', 'Comedy', 'Horror']","['sexploitation', 'maid', 'seductress', 'voyeu...","['Donatella Damiani', 'Scott Coffey', 'Gabriel...",0.134678
3,Paprika,When a machine that allows therapists to enter...,"['Science Fiction', 'Thriller', 'Animation']","['research', 'japan', 'dreams', 'based on nove...","['Megumi Hayashibara', 'Tohru Emori', 'Katsuno...",0.131386
4,A Nightmare on Elm Street Part 2: Freddy's Rev...,A teenage boy is haunted in his dreams by dece...,['Horror'],"['high school', 'fire', 'dreams', 'nightmare',...","['Robert Englund', 'Mark Patton', 'Kim Myers',...",0.126651


## 6. Enfoque semántico: embeddings (Sentence-BERT)

Usamos el modelo preentrenado `all-mpnet-base-v2`, con soporte para hasta **514 tokens** y vectores de **768 dimensiones**, para convertir cada texto en un vector denso que captura su significado. La recuperación se realiza por similitud coseno entre el embedding de la consulta y los de las películas.

In [87]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

/Users/lucas/anaconda3/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### 6.1. Validación del límite de tokens (evitar truncamiento)

Los modelos basados en Transformers tienen una longitud máxima de secuencia de entrada. Para el modelo `all-MiniLM-L6-v2`, el límite de contexto es de **256 tokens**.

Si un texto enriquecido (`search_text`) supera este límite, el modelo lo trunca de manera silenciosa al generar los embeddings, descartando la información final (por ejemplo, parte del *overview* o el reparto). A continuación, validamos cuántos textos de nuestro dataset exceden este límite y mostramos algunas estadísticas de interés.

In [88]:
# 1. Obtener el límite de secuencia configurado en el modelo
max_seq_len = model.max_seq_length
print(f"Límite máximo de secuencia del modelo: {max_seq_len} tokens\n")

# 2. Tokenizar cada texto y calcular su longitud (incluyendo tokens especiales)
tokenizer = model.tokenizer
token_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in movie_texts]

max_len      = max(token_lengths)
mean_len     = np.mean(token_lengths)
median_len   = np.median(token_lengths)
exceed_count = sum(1 for l in token_lengths if l > max_seq_len)
pct_exceed   = (exceed_count / len(token_lengths)) * 100

print("Estadísticas de tokens para 'search_text' (keywords limitados a 8):")
print(f"- Longitud máxima registrada: {max_len} tokens")
print(f"- Longitud promedio:          {mean_len:.2f} tokens")
print(f"- Longitud mediana:           {median_len:.2f} tokens")
print(f"- Documentos que superan el límite ({max_seq_len} tokens): {exceed_count} ({pct_exceed:.2f}%)")

if exceed_count == 0:
    print(f"\n✅ Ningún texto supera el límite de {max_seq_len} tokens. Sin truncamiento.")
else:
    print(f"\n⚠️ {exceed_count} textos superan el límite y serán truncados por el modelo.")

Límite máximo de secuencia del modelo: 384 tokens

Estadísticas de tokens para 'search_text' (keywords limitados a 8):
- Longitud máxima registrada: 315 tokens
- Longitud promedio:          163.16 tokens
- Longitud mediana:           160.50 tokens
- Documentos que superan el límite (384 tokens): 0 (0.00%)

✅ Ningún texto supera el límite de 384 tokens. Sin truncamiento.


In [89]:
movie_embeddings = model.encode(
    movie_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Shape de la matriz de embeddings:", movie_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Shape de la matriz de embeddings: (996, 768)


In [90]:
def search_semantic(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, movie_embeddings)[0]
    top_idx = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_idx][["title", "overview", "genres", "keywords", "cast"]].copy()
    results["similarity"] = similarities[top_idx]
    return results.reset_index(drop=True)

## 7. Evaluación cualitativa

Inspeccionamos los resultados del buscador semántico sobre consultas en lenguaje natural para verificar la coherencia de las películas recuperadas.

In [91]:
queries = [
    "animated movie with dinosaurs and friendship",
    "historical drama about royalty and power",
    "film about genius, science and war",
    "movie about dreams, memory and the mind",
]

for q in queries:
    print("=" * 100)
    print("Consulta:", q)
    display(search_semantic(q, top_k=3))

Consulta: animated movie with dinosaurs and friendship


,title,overview,genres,keywords,cast,similarity
0,The Good Dinosaur,An epic journey into the world of dinosaurs wh...,"['Adventure', 'Animation', 'Family']","['friendship', 'tyrannosaurus rex', 'cartoon',...","['Frances McDormand', 'Raymond Ochoa', 'Jeffre...",0.702531
1,Doraemon: Nobita's Dinosaur,When Suneo does not show Nobita the fossil of ...,"['Animation', 'Family', 'Adventure', 'Science ...","['pet', 'time travel', 'dinosaur', 'based on m...","['Nobuyo Oyama', 'Noriko Ohara', 'Michiko Nomu...",0.593055
2,The Land Before Time,An orphaned brontosaurus named Littlefoot sets...,"['Family', 'Animation', 'Adventure']","['vulkan', 'loss of loved one', 'tyrannosaurus...","['Gabriel Damon', 'Candace Hutson', 'Will Ryan...",0.574003


Consulta: historical drama about royalty and power


,title,overview,genres,keywords,cast,similarity
0,"Red, White & Royal Blue","After an altercation between Alex, the preside...","['Comedy', 'Romance']","['based on novel or book', 'politics', 'prince...","['Taylor Zakhar Perez', 'Nicholas Galitzine', ...",0.586467
1,Henry V,"In 1415, in the midst of the Hundred Years' Wa...","['War', 'Drama', 'History']","['france', 'kingdom', 'theater play', 'based o...","['Kenneth Branagh', 'Derek Jacobi', 'Brian Ble...",0.546946
2,The Prince of Egypt,The strong bond between two Royal Egyptian bro...,"['Adventure', 'Animation', 'Drama', 'Family']","['egypt', 'moses', 'kingdom', 'pyramid', 'exod...","['Val Kilmer', 'Ralph Fiennes', 'Michelle Pfei...",0.531194


Consulta: film about genius, science and war


,title,overview,genres,keywords,cast,similarity
0,Tropic Thunder,A group of self-absorbed actors set out to mak...,"['Action', 'Comedy', 'Adventure', 'War']","['vietnam', 'movie business', 'satire', 'parod...","['Ben Stiller', 'Robert Downey Jr.', 'Jack Bla...",0.553253
1,Independence Day,Strange phenomena surface around the globe. Th...,"['Action', 'Adventure', 'Science Fiction']","['spacecraft', 'showdown', 'independence', 'pa...","['Will Smith', 'Bill Pullman', 'Jeff Goldblum'...",0.541733
2,Starship Troopers,"Set in the future, the story follows a young s...","['Adventure', 'Action', 'Thriller', 'Science F...","['spacecraft', 'space marine', 'army', 'moon',...","['Casper Van Dien', 'Dina Meyer', 'Denise Rich...",0.518812


Consulta: movie about dreams, memory and the mind


,title,overview,genres,keywords,cast,similarity
0,In Your Dreams,Stevie and her little brother Elliot journey i...,"['Adventure', 'Animation', 'Comedy', 'Family',...","['magic', 'wish', 'family', 'dream']","['Jolie Hoang-Rappaport', 'Elias Janssen', 'Si...",0.584734
1,Paprika,When a machine that allows therapists to enter...,"['Science Fiction', 'Thriller', 'Animation']","['research', 'japan', 'dreams', 'based on nove...","['Megumi Hayashibara', 'Tohru Emori', 'Katsuno...",0.570329
2,Inception,"Cobb, a skilled thief who commits corporate es...","['Action', 'Science Fiction', 'Adventure']","['mission', 'dreams', 'kidnapping', 'spy', 'al...","['Leonardo DiCaprio', 'Joseph Gordon-Levitt', ...",0.551643


## 8. Evaluación cuantitativa: Hit Rate@K y MRR

Definimos un conjunto de consultas sintéticas, cada una con su película esperada (*ground truth*), diseñadas para simular búsquedas reales basadas en paráfrasis, sinónimos y descripciones parciales. Medimos:

- **Hit Rate@K:** proporción de consultas cuya película esperada aparece dentro del Top-K (métrica binaria por consulta).
- **MRR (Mean Reciprocal Rank):** promedio del inverso de la posición del resultado correcto (1 si aparece primero, 0.5 si segundo, etc.), midiendo la precisión posicional del sistema.

In [92]:
# Conjunto de evaluación: (consulta sintética, título esperado)
evaluation_queries = [
    ("child prodigy with telekinesis directed by Danny DeVito", "Matilda"),
    ("Harrison Ford and Sean Connery search for the Holy Grail", "Indiana Jones and the Last Crusade"),
    ("horror sequel possessed doll in toy factory", "Child's Play 2"),
    ("alien time loop Tom Cruise", "Edge of Tomorrow"),
    ("James Bond playing poker in Montenegro Daniel Craig", "Casino Royale"),
    ("Vietnam veteran with low IQ Tom Hanks", "Forrest Gump"),
    ("Jason Momoa and Jack Black trapped in a cubic world", "A Minecraft Movie"),
    ("boy reads a magical book about a luck dragon", "The NeverEnding Story"),
    ("stuntman searches for missing movie star Ryan Gosling", "The Fall Guy"),
    ("Al Pacino seeks forgiveness as Michael Corleone", "The Godfather Part III"),
    ("mutant mercenary who breaks the fourth wall Ryan Reynolds", "Deadpool 2"),
    ("monsters from another dimension destroy the world 2025 Milla Jovovich", "Worldbreaker"),
    ("Cillian Murphy atomic bomb Nolan", "Oppenheimer"),
    ("Peter Parker multiverse Doctor Strange", "Spider-Man: No Way Home"),
    ("thief who infiltrates dreams DiCaprio", "Inception"),
    ("cyberpunk sequel Keanu Reeves the One", "The Matrix Reloaded"),
    ("Brad Pitt and Morgan Freeman 7 deadly sins killer", "Se7en"),
    ("black and white movie Liam Neeson saving Jews", "Schindler's List"),
    ("freed slave western looking for his wife Jamie Foxx", "Django Unchained"),
    ("Uma Thurman assassin with samurai sword revenge", "Kill Bill: Vol. 1"),
    ("Irish mob infiltrators in Boston DiCaprio Matt Damon", "The Departed"),
    ("Christian Bale vs Heath Ledger as the Joker", "The Dark Knight"),
    ("romance on a ship 1912 DiCaprio and Kate Winslet", "Titanic"),
    ("lion exiled by his uncle Scar", "The Lion King"),
    ("Matthew McConaughey travels through a wormhole", "Interstellar"),
    ("New York ballerina obsessed with perfection Natalie Portman", "Black Swan"),
    ("ogre travels to Far Far Away to meet his in-laws", "Shrek 2"),
    ("arrogant race car gets lost on Route 66", "Cars"),
    ("Disney lion reclaims throne after Mufasa's death", "The Lion King"),
    ("pet chameleon becomes sheriff Johnny Depp", "Rango"),
    ("alien supervillain creates a new hero Will Ferrell", "Megamind"),
    ("ordinary Lego figure has to save the universe", "The Lego Movie"),
    ("Madrigal family in Colombia girl without magic", "Encanto"),
    ("emotions get lost in the mind of an 11-year-old girl", "Inside Out"),
    ("stop-motion thief fox against three farmers Wes Anderson", "Fantastic Mr. Fox"),
    ("Miles Morales teams up with other Spider-Men from different dimensions", "Spider-Man: Into the Spider-Verse"),
    ("old man who flies his house with balloons and a boy scout", "Up"),
    ("clownfish looking for his son in a dentist's aquarium", "Finding Nemo"),
    ("young Chinese woman disguises herself as a soldier guardian dragon", "Mulan"),
    ("bunny cop and con artist fox investigate mystery", "Zootopia"),
    ("supervillain adopts three girls to steal the moon", "Despicable Me"),
    ("time loop fighting aliens", "Edge of Tomorrow"),
    ("little girl telekinesis", "Matilda"),
    ("finding the holy grail", "Indiana Jones and the Last Crusade"),
    ("killer doll toy factory", "Child's Play 2"),
    ("mafia part 3", "The Godfather Part III"),
    ("ballet dancer thriller", "Black Swan"),
    ("horror cant make noise", "A Quiet Place"),
    ("skeleton pirates curse", "Pirates of the Caribbean: The Curse of the Black Pearl"),
    ("spider multiverse", "Spider-Man: No Way Home"),
    ("red race car route 66", "Cars"),
    ("ogre sequel in laws", "Shrek 2"),
    ("ship sink romance", "Titanic"),
    ("stealing dreams heist", "Inception"),
    ("saving jews ww2", "Schindler's List"),
    ("matrix sequel zion", "The Matrix Reloaded"),
]

print("Consultas de evaluación:", len(evaluation_queries))

Consultas de evaluación: 56


In [93]:
def evaluate_system(search_function, queries, top_k=5):
    """Evalúa un sistema de búsqueda devolviendo Hit Rate@K y MRR."""
    hits = 0
    reciprocal_ranks = []

    for query, expected_title in queries:
        retrieved_titles = search_function(query, top_k=top_k)["title"].tolist()

        if expected_title in retrieved_titles:
            hits += 1
            rank = retrieved_titles.index(expected_title) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    hit_rate = hits / len(queries)
    mrr = np.mean(reciprocal_ranks)
    return hit_rate, mrr

In [94]:
print("Evaluando TF-IDF (léxico)...")
tfidf_hit, tfidf_mrr = evaluate_system(search_tfidf, evaluation_queries, top_k=5)

print("Evaluando Embeddings (semántico)...")
sem_hit, sem_mrr = evaluate_system(search_semantic, evaluation_queries, top_k=5)

comparison_df = pd.DataFrame({
    "Modelo": ["TF-IDF (Léxico)", "Embeddings (Semántico)"],
    "Hit Rate@5": [f"{tfidf_hit:.2%}", f"{sem_hit:.2%}"],
    "MRR": [round(tfidf_mrr, 4), round(sem_mrr, 4)],
})

print("\n--- Resultados de la Evaluación ---")
comparison_df

Evaluando TF-IDF (léxico)...
Evaluando Embeddings (semántico)...

--- Resultados de la Evaluación ---


,Modelo,Hit Rate@5,MRR
0,TF-IDF (Léxico),82.14%,0.7438
1,Embeddings (Semántico),96.43%,0.8458


## 9. Evaluación con relevancia múltiple: P@K, MAP y nDCG@K

La evaluación anterior sigue el modelo de **búsqueda por ítem conocido** (*known-item search*): cada consulta tiene exactamente una película esperada como respuesta correcta. Este es el criterio más estricto —y el más común en benchmarks de IR— pero no captura bien la naturaleza del caso de uso real.

En un buscador de películas, múltiples títulos pueden ser igualmente válidos para una misma consulta. Un usuario que busca *"película animada sobre dinosaurios y amistad"* puede quedar satisfecho con *The Good Dinosaur*, *The Land Before Time* o cualquier otra película del género. Ignorar esta multi-relevancia subestima sistemáticamente ambos sistemas.

### Diseño de los *relevance judgments* (qrels)

Construimos un conjunto de juicios de relevancia **graduada** para cada consulta siguiendo el enfoque de colección de prueba (*test collection*) del estándar Cranfield/TREC:

| Grado | Criterio |
|---|---|
| **2** — Altamente relevante | La película específica del *ground truth* |
| **1** — Parcialmente relevante | Películas que comparten ≥ 2 géneros con la película del *ground truth* |
| **0** — No relevante | El resto del catálogo |

### Métricas implementadas

- **P@10 (Precision at 10):** fracción de los 10 resultados recuperados que son relevantes (grado ≥ 1). Métrica simple y directamente interpretable.
- **MAP (Mean Average Precision):** promedio de la *Average Precision* sobre todas las consultas, calculada como la media de la precisión en cada posición en la que aparece un documento relevante, normalizada por el total de documentos relevantes en la colección. Valores bajos son esperables cuando la colección relevante es grande y K es pequeño: si hay ~160 documentos relevantes y se recuperan solo 10, el MAP máximo teórico es ≈ 0.063.
- **nDCG@10 (Normalized Discounted Cumulative Gain):** acumula la ganancia de relevancia (0, 1, 2) de los resultados recuperados, aplicando un descuento logarítmico por posición y normalizando por el ranking ideal. Captura tanto **cuántos** documentos relevantes se recuperan como **en qué posición** aparecen y **cuán relevantes** son.

In [95]:
import math

def parse_genres(val):
    try:
        result = ast.literal_eval(val)
        return set(result) if isinstance(result, list) else set()
    except (ValueError, SyntaxError):
        return set()

df["genres_set"] = df["genres"].apply(parse_genres)
title_to_genres = df.set_index("title")["genres_set"].to_dict()

def build_qrels(queries, grade_gold=2, grade_partial=1, min_shared=2):
    """
    Para cada query construye un dict {título: grado_de_relevancia}.
    - grade_gold: película esperada del ground truth
    - grade_partial: películas con ≥ min_shared géneros en común con el gold
    """
    qrels = {}
    for query, gold in queries:
        gold_genres = title_to_genres.get(gold, set())
        rel = {}
        for title, genres in title_to_genres.items():
            if title == gold:
                rel[title] = grade_gold
            elif len(gold_genres & genres) >= min_shared:
                rel[title] = grade_partial
        qrels[query] = rel
    return qrels

qrels = build_qrels(evaluation_queries)

# Estadísticas de la colección de prueba
rel_per_query = [sum(1 for g in q.values() if g >= 1) for q in qrels.values()]
grade2_per_query = [sum(1 for g in q.values() if g == 2) for q in qrels.values()]
grade1_per_query = [sum(1 for g in q.values() if g == 1) for q in qrels.values()]

print("Estadísticas de la colección de prueba (qrels):")
print(f"  Consultas totales:               {len(qrels)}")
print(f"  Docs relevantes por consulta:    min={min(rel_per_query)}  max={max(rel_per_query)}  media={np.mean(rel_per_query):.1f}")
print(f"  Grado 2 (gold exacto):           siempre {np.mean(grade2_per_query):.0f}")
print(f"  Grado 1 (≥2 géneros en común):   media={np.mean(grade1_per_query):.1f}")

Estadísticas de la colección de prueba (qrels):
  Consultas totales:               56
  Docs relevantes por consulta:    min=1  max=294  media=159.9
  Grado 2 (gold exacto):           siempre 1
  Grado 1 (≥2 géneros en común):   media=158.9


In [96]:
def precision_at_k(retrieved, qrels_q, k):
    """Fracción de los top-K recuperados que son relevantes (grado ≥ 1)."""
    return sum(1 for t in retrieved[:k] if qrels_q.get(t, 0) >= 1) / k

def average_precision(retrieved, qrels_q):
    """AP para una sola consulta. Normaliza por el total de docs relevantes en la colección."""
    n_rel = sum(1 for g in qrels_q.values() if g >= 1)
    if n_rel == 0:
        return 0.0
    hits, ap = 0, 0.0
    for i, title in enumerate(retrieved, 1):
        if qrels_q.get(title, 0) >= 1:
            hits += 1
            ap += hits / i
    return ap / n_rel

def ndcg_at_k(retrieved, qrels_q, k):
    """nDCG@K con relevancia graduada (grados 0, 1, 2)."""
    dcg  = sum(qrels_q.get(t, 0) / math.log2(i + 2) for i, t in enumerate(retrieved[:k]))
    ideal_grades = sorted(qrels_q.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal_grades))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_multi_relevant(search_fn, queries, qrels_dict, k=10):
    """Evalúa un sistema devolviendo P@K, MAP y nDCG@K sobre relevancia múltiple."""
    P, AP, nDCG = [], [], []
    for query, _ in queries:
        retrieved = search_fn(query, top_k=k)["title"].tolist()
        q = qrels_dict[query]
        P.append(precision_at_k(retrieved, q, k))
        AP.append(average_precision(retrieved, q))
        nDCG.append(ndcg_at_k(retrieved, q, k))
    return {f"P@{k}": np.mean(P), "MAP": np.mean(AP), f"nDCG@{k}": np.mean(nDCG)}

In [97]:
print("Evaluando TF-IDF (relevancia múltiple)...")
tfidf_multi = evaluate_multi_relevant(search_tfidf, evaluation_queries, qrels, k=10)

print("Evaluando Embeddings (relevancia múltiple)...")
sem_multi = evaluate_multi_relevant(search_semantic, evaluation_queries, qrels, k=10)

comparison_multi = pd.DataFrame({
    "Modelo": ["TF-IDF (Léxico)", "Embeddings (Semántico)"],
    "P@10":      [round(tfidf_multi["P@10"],   4), round(sem_multi["P@10"],   4)],
    "MAP":       [round(tfidf_multi["MAP"],     4), round(sem_multi["MAP"],     4)],
    "nDCG@10":   [round(tfidf_multi["nDCG@10"],4), round(sem_multi["nDCG@10"],4)],
})

print("\n--- Resultados con Relevancia Múltiple (K=10) ---")
print("  (MAP bajo es esperable: ~160 docs relevantes pero sólo se recuperan 10)")
comparison_multi

Evaluando TF-IDF (relevancia múltiple)...
Evaluando Embeddings (relevancia múltiple)...

--- Resultados con Relevancia Múltiple (K=10) ---
  (MAP bajo es esperable: ~160 docs relevantes pero sólo se recuperan 10)


,Modelo,P@10,MAP,nDCG@10
0,TF-IDF (Léxico),0.4018,0.0543,0.5471
1,Embeddings (Semántico),0.5179,0.0630,0.6571


## 10. Conclusiones

Se realizaron dos evaluaciones cuantitativas complementarias sobre las 56 consultas con película esperada (*ground truth*), que en conjunto ofrecen una visión más completa del comportamiento de cada sistema.

### Evaluación .8 — Búsqueda por ítem conocido (Hit Rate@5 y MRR)

| Modelo | Hit Rate@5 | MRR |
|---|---|---|
| TF-IDF (léxico) | 82.14% | 0.7438 |
| Embeddings (semántico) | **96.43%** | **0.8458** |

### Evaluación .9 — Relevancia múltiple (P@10, MAP y nDCG@10)

| Modelo | P@10 | MAP | nDCG@10 |
|---|---|---|---|
| TF-IDF (léxico) | 0.4018 | 0.0543 | 0.5471 |
| Embeddings (semántico) | **0.5179** | **0.0630** | **0.6571** |

### Análisis integrado

- El enfoque semántico basado en `all-mpnet-base-v2` supera al léxico en **todas** las métricas de ambas evaluaciones, lo que confirma de forma robusta su ventaja independientemente del criterio de éxito utilizado.

- En la evaluación de ítem conocido, el modelo semántico mejora el Hit Rate@5 en **+14.29 pp** y el MRR en **+0.102**, lo que indica que no solo recupera la película correcta con mayor frecuencia, sino que la posiciona más arriba en el ranking.

- En la evaluación de relevancia múltiple, el P@10 pasa de 0.40 a 0.52 (+29 %): en promedio, el modelo semántico recupera una película de género relevante adicional por cada 10 resultados frente al TF-IDF. El nDCG@10 refleja además que esas películas tienden a aparecer en posiciones más altas del ranking (0.55 → 0.66, +20 %).

- El MAP es bajo en ambos modelos (≈ 0.05–0.06), pero esto es **esperable por construcción**: con ~160 documentos relevantes en la colección y solo 10 recuperados, el MAP máximo teórico alcanzable es ≈ 0.063. Lo relevante es que incluso en este escenario desfavorable el modelo semántico supera al léxico.

- El enfoque **TF-IDF** depende de coincidencias léxicas exactas: cuando la consulta usa paráfrasis o sinónimos, su capacidad para recuperar y posicionar documentos relevantes se degrada visiblemente en las dos evaluaciones.

- La evaluación cualitativa (§7) es coherente: consultas como *"film about genius, science and war"* o *"movie about dreams, memory and the mind"* recuperan películas pertinentes (Oppenheimer, Inception, Paprika) pese a no compartir las palabras exactas de la sinopsis.

### Limitaciones

- El dataset se limita a 996 películas tras eliminar duplicados; algunas consultas del set de evaluación referencian títulos ausentes, penalizando a ambos modelos por igual.
- Los *relevance judgments* de §9 son un *silver standard* derivado automáticamente de los géneros TMDB, no de jueces humanos.
- El modelo `all-mpnet-base-v2` está optimizado para inglés; consultas en español o con mezcla de idiomas pueden degradar su rendimiento.

### Trabajo futuro

- Incorporar un modelo multilingüe (`paraphrase-multilingual-mpnet-base-v2`) para soportar consultas en español.
- Ampliar el dataset vía la API de TMDB para reducir el efecto de los títulos ausentes.
- Construir *relevance judgments* humanos para un subconjunto de consultas y comparar con el *silver standard* de géneros.
- Escalar la búsqueda semántica con un índice vectorial (FAISS) para colecciones de mayor tamaño.